In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Importando as classes das suas respectivas pastas
from planta.paciente import Paciente
from identificador.rls import Identificador
from controladores.gpc_fixo import Controlador_GPC_Fixo
from controladores.gpc_adaptativo import Controlador_GPC_Adaptativo
from controladores.pertubacao import Controlador_32_Perturbacao
from controladores.restricoes import Controlador_33_Restricoes
from controladores.idc import Controlador_35_Inovacoes

#COMPARATIVO

def run_simulation(ro, lambda_val, P0, map_ref, sigma,
                   a1_ini, b1_ini, bm_ini, d_ini, m_ini,
                   k_change, a1_new, b1_new, bm_new, d_new, m_new,
                   N_MC=50):

    N     = 500
    yref  = P0 - map_ref
    d_vetor = [2, 3, 4, 5]
    m_vetor = [2, 3, 4, 5]
    teta_real_ini = [a1_ini, b1_ini, bm_ini]
    t    = np.arange(N)
    pref = np.full(N, map_ref)

    # Configuração dos 5 controladores
    # make_ctrl é uma lambda para recriar o objeto a cada run MC
    cfg = [
    ('GPC Fixo',        'seagreen',
     lambda: Controlador_GPC_Fixo([a1_ini, b1_ini, bm_ini], d_ini, m_ini, ro)),
    ('GPC Adaptativo',  'mediumpurple',
     lambda: Controlador_GPC_Adaptativo(ro)),
    ('3.2 Perturbação', 'royalblue',
     lambda: Controlador_32_Perturbacao(amplitude=8.0, taxa_decaimento=0.997, seed=None)),
    ('3.3 Restrições',  'crimson',
     lambda: Controlador_33_Restricoes(u_lim=15.0)),
    ('3.5 IDC',         'darkorange',
     lambda: Controlador_35_Inovacoes(lambda_idc=0.5)),   # <- aqui estava alpha=3.0, beta=0.9
    ]

    #  Acumuladores
    res = {}
    for nome, cor, _ in cfg:
        res[nome] = dict(
            cor          = cor,
            all_map      = np.zeros((N_MC, N)),
            all_i        = np.zeros((N_MC, N)),
            # guardamos a última run para gráficos não-estocásticos
            delay_h      = np.zeros(N),
            delay_real   = np.zeros(N),
            teta_all     = np.zeros((N, 12)),
            teta_real    = np.zeros((N, 3)),
            custos       = np.zeros((N, 4)),
            incerteza    = np.zeros((N, 4)),
        )

    # Loop Monte Carlo
    for nome, _, make_ctrl in cfg:
        r = res[nome]
        for mc in range(N_MC):
            paciente      = Paciente(teta_real_ini, d_ini, m_ini, P0, sigma)
            identificador = Identificador(lambda_val, d_vetor, m_vetor, N)
            ctrl          = make_ctrl()

            map_h  = np.zeros(N);  i_h   = np.zeros(N)
            d_h    = np.zeros(N);  dr_h  = np.zeros(N)
            ta_h   = np.zeros((N, 12));  tr_h = np.zeros((N, 3))
            cu_h   = np.zeros((N, 4));   inc_h = np.zeros((N, 4))

            I_pac = 0.0

            # DEPOIS — rampa de 50 passos
            RAMPA = 50  # passos para a transição

            for k in range(N):
                # Rampa gradual dos parâmetros
                if k_change <= k < k_change + RAMPA:
                    alpha = (k - k_change) / RAMPA
                    paciente.teta_real = np.array([
                        a1_ini + alpha * (a1_new - a1_ini),
                        b1_ini + alpha * (b1_new - b1_ini),
                        bm_ini + alpha * (bm_new - bm_ini),
                    ])

                # Mudança completa só ao fim da rampa
                elif k == k_change + RAMPA:
                    paciente.teta_real = np.array([a1_new, b1_new, bm_new])
                    paciente.d = int(d_new)
                    paciente.m = int(m_new)
                    identificador.Pant = np.kron(np.eye(4), 10 * np.eye(3))

                tr_h[k] = paciente.teta_real;  dr_h[k] = paciente.d

                MAP, P_med, _ = paciente.step(I_pac)
                teta_est, d_est, m_est, vetor_inc, custos_step, P_melhor = identificador.RLS(P_med, I_pac)
                var_b1  = vetor_inc[identificador.ind_melhor]
                I_atual = ctrl.calcular_controle(yref, P_med, teta_est, d_est, m_est, var_b1, P_melhor)

                I_pac    = I_atual
                map_h[k] = MAP;     i_h[k]  = I_atual
                d_h[k]   = d_est;   ta_h[k] = identificador.teta_ea
                cu_h[k]  = custos_step;  inc_h[k] = vetor_inc

            r['all_map'][mc] = map_h;  r['all_i'][mc] = i_h
            # sobrescreve — guarda a última run
            r['delay_h']   = d_h;   r['delay_real'] = dr_h
            r['teta_all']  = ta_h;  r['teta_real']  = tr_h
            r['custos']    = cu_h;  r['incerteza']  = inc_h

    # Funções auxiliares de plot
    def _vline(ax):
        ax.axvline(x=k_change, color='black', linestyle=':', linewidth=1.2,
                   label='Mudança paciente')

    def _envelope(ax, nome, arr_mc):
        cor   = res[nome]['cor']
        media = arr_mc.mean(axis=0)
        ax.plot(t, media, color=cor, linewidth=2, label=nome)

    # FIGURA 1 — MAP, Infusão, Atraso, a1, b1, bm

    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle(f'Comparativo Monte Carlo — 5 Controladores  (N_MC={N_MC})',
                 fontsize=14, fontweight='bold')

    # MAP
    ax = axes[0, 0]
    for nome, _, _ in cfg:
        _envelope(ax, nome, res[nome]['all_map'])
    ax.plot(t, pref, 'r--', linewidth=1.5, label=f'Ref ({map_ref} mmHg)')
    _vline(ax)
    ax.set_ylim(50, 200);  ax.set_title('Pressão Arterial — mmHg')
    ax.legend(fontsize=8, loc='upper right');  ax.grid(True)

    # Infusão
    ax = axes[1, 0]
    for nome, _, _ in cfg:
        _envelope(ax, nome, res[nome]['all_i'])
    _vline(ax)
    ax.set_ylim(-10, 190);  ax.set_title('Taxa de Infusão de SNP — ml/h')
    ax.legend(fontsize=8, loc='upper right');  ax.grid(True)

    # Parâmetro a1
    ax = axes[0, 1]
    nome_ref = 'GPC Fixo'  # referência explícita para o valor real
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 0], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 0], 'k--', linewidth=2, label='a1 Real')
    _vline(ax)
    ax.set_ylim(-1.5, 0.5);  ax.set_title('Parâmetro a1')
    ax.legend(fontsize=8);  ax.grid(True)

    # Parâmetro b1
    ax = axes[1, 1]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 1], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 1], 'k--', linewidth=2, label='b1 Real')
    _vline(ax)
    ax.set_ylim(-0.5, 1.5);  ax.set_title('Parâmetro b1')
    ax.legend(fontsize=8);  ax.grid(True)

    # Parâmetro bm
    ax = axes[2, 1]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['teta_all'][:, 2], color=res[nome]['cor'],
                linewidth=1.2, alpha=0.85, label=nome)
    ax.plot(t, res[nome_ref]['teta_real'][:, 2], 'k--', linewidth=2, label='bm Real')
    _vline(ax)
    ax.set_ylim(-1.0, 1.5);  ax.set_title('Parâmetro bm')
    ax.legend(fontsize=8);  ax.grid(True)

    # Atraso
    ax = axes[2, 0]
    for nome, _, _ in cfg:
        ax.plot(t, res[nome]['delay_h'], color=res[nome]['cor'],
                linewidth=1.5, label=nome)
    ax.plot(t, res[nome_ref]['delay_real'], 'k--', linewidth=2, label='Atraso Real')
    _vline(ax)
    ax.set_ylim(1, 6);  ax.set_title('Atraso Estimado vs Real')
    ax.legend(fontsize=8);  ax.grid(True)

    plt.tight_layout()
    plt.show()

    # FIGURA 2 — Custo Acumulado (1 subplot por controlador)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    # 5 controladores + 1 célula vazia na grade 2×3
    fig.suptitle('Custo Acumulado por Modelo — última run', fontsize=13, fontweight='bold')
    labels_d = ['d=2', 'd=3', 'd=4', 'd=5']
    cores_d  = ['gray', 'blue', 'green', 'red']
    axs_flat = axes.flat
    for nome, _, _ in cfg:
        ax = next(axs_flat)
        for j, (lbl, cd) in enumerate(zip(labels_d, cores_d)):
            ax.semilogy(t, res[nome]['custos'][:, j], color=cd,
                        label=lbl, alpha=0.8, linewidth=1.3)
        ax.axvline(x=k_change, color='k', linestyle='--', linewidth=1)
        ax.set_title(nome, color=res[nome]['cor'], fontweight='bold', fontsize=11)
        ax.set_ylabel('Custo (log)');  ax.legend(fontsize=8);  ax.grid(True)
    next(axs_flat).set_visible(False)  # célula vazia
    plt.tight_layout()
    plt.show()

    # FIGURA 3 — Variância de b1 (Incerteza) por modelo
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('Variância do Parâmetro b1 (Incerteza) por Modelo — última run',
                 fontsize=13, fontweight='bold')
    labels_m = ['Atraso 2', 'Atraso 3', 'Atraso 4', 'Atraso 5']
    cores_m  = ['blue', 'orange', 'green', 'red']
    axs_flat = axes.flat
    for nome, _, _ in cfg:
        ax = next(axs_flat)
        for i, (lbl, cm) in enumerate(zip(labels_m, cores_m)):
            ax.plot(t, res[nome]['incerteza'][:, i], color=cm,
                    label=lbl, linewidth=1.5)
        ax.axvline(x=k_change, color='k', linestyle='--', linewidth=1)
        ax.set_title(nome, color=res[nome]['cor'], fontweight='bold', fontsize=11)
        ax.set_yscale('log');  ax.set_ylabel('Variância')
        ax.legend(fontsize=8);  ax.grid(True, which='both', ls='-', alpha=0.4)
    next(axs_flat).set_visible(False)
    plt.tight_layout()
    plt.show()
